In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score



In [ ]:
df = pd.read_csv('/Users/fabianatorres/Downloads/archive/Malware_and_benign_recognition.csv')

In [ ]:
df.head()

In [ ]:
#EDA
print(df['Malicious'].value_counts()) #how many Malicious vs Benign

In [ ]:
import seaborn as sns

sns.set_style('darkgrid') #color
plt.figure(figsize=(6,4)) #size
plt.pie(df['Malicious'].value_counts(), labels = df['Malicious'].value_counts().index, autopct='%1.1f%%')
plt.title('Benign vs Malicious')

In [ ]:
x= df.drop(columns=["File", "Malicious"]) #this will hold all the the columns except Malicious and File

y=df['Malicious']

In [ ]:
to_plot = [
    'SizeOfCode', 'SizeOfInitializedData', 
         'SizeOfUninitializedData', 'AddressOfEntryPoint', 
         'BaseOfData', 'ImageBase', 'NumberOfSections', 
         'DllCharacteristics'
]

x = df[to_plot]
y = df['Malicious']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42) 
print(f"{len(x)} - number of elements in the entire dataset.")
print(f"{len(x_train)} - number of training elements.")
print(f"{len(x_test)} - number of testing elements.")
print(f"{len(x_train) + len(x_test)} - sum of training and testing elements.")

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test) 

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import Aer
from qiskit_aer import AerSimulator


service = QiskitRuntimeService(channel="local")

backend = AerSimulator() #telling that we are using a simulator


In [ ]:
from qiskit.circuit.library import zz_feature_map

# creating the feature map
num_features = x_train.shape[1]

feature_map = zz_feature_map(feature_dimension=num_features, reps=1)
feature_map.decompose().draw(output="mpl", style="clifford", fold=20)

from qiskit_aer.primitives import SamplerV2 as AerSampler
sampler = AerSampler.from_backend(backend)

from qiskit_algorithms.state_fidelities import ComputeUncompute
fidelity = ComputeUncompute(sampler=sampler)

# Kernel
from qiskit_machine_learning.kernels import TrainableFidelityQuantumKernel
quantum_kernel = TrainableFidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)

feature_map.decompose().draw()

num_features = x_train.shape[1]

#feature_map = zz_feature_map(feature_dimension=num_features, reps=1)
#feature_map.decompose().draw(output="mpl", style="clifford", fold=20)

In [ ]:
K_train = quantum_kernel.evaluate(x_train, x_train)

qsvm_model = SVC(kernel='precomputed', C=1.0, random_state=42)
qsvm_model.fit(K_train, y_train)

K_test = quantum_kernel.evaluate(x_test, x_train)

# Predict the labels for the test data

y_pred = qsvm_model.predict(K_test)

# Calc accuracy and roc auc
acc = accuracy_score(qy_test, y_pred)
auc = roc_auc_score(qy_test, y_pred)

print(f"\nAccuracy: {acc:.4f}")
print(f"ROC AUC:  {auc:.4f}\n")
print(classification_report(qy_test, y_pred, target_names=['Benign (0)', 'Malware (1)']))